# Assignment 2 — ASR Decoding Evaluation

Запускай ячейки по порядку. Полная оценка (200 сэмплов, все методы) займёт ~2–3 часа на CPU или ~20–30 мин на GPU T4.

**Быстрая проверка:** поставь `MAX_SAMPLES = 10` в ячейке конфигурации.

In [ ]:
# ── 1. Установка зависимостей ──────────────────────────────────────────────
!pip install -q https://github.com/kpu/kenlm/archive/master.zip
!pip install -q jiwer transformers torch torchaudio soundfile pandas matplotlib

In [ ]:
# ── 2. Клонирование репозитория ────────────────────────────────────────────
# Замени URL на адрес своего публичного репозитория на GitHub
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # <- ЗАМЕНИТЬ

!git clone {REPO_URL} repo
%cd repo/assignments/assignment2

In [ ]:
# ── 3. Распаковка LM (если .arpa.gz) ──────────────────────────────────────
import os, gzip, shutil

gz_path = 'lm/3-gram.pruned.1e-7.arpa.gz'
arpa_path = 'lm/3-gram.pruned.1e-7.arpa'

if os.path.exists(gz_path) and not os.path.exists(arpa_path):
    print('Decompressing LM...')
    with gzip.open(gz_path, 'rb') as f_in, open(arpa_path, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
    print('Done.')
elif os.path.exists(arpa_path):
    print('LM already decompressed.')
else:
    print('ERROR: LM file not found!')

In [ ]:
# ── 4. Конфигурация ────────────────────────────────────────────────────────
MAX_SAMPLES = 200   # поставь 10 для быстрой проверки
BEAM_WIDTH  = 10
LM_PATH     = 'lm/3-gram.pruned.1e-7.arpa'

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}  |  MAX_SAMPLES: {MAX_SAMPLES}')

In [ ]:
# ── 5. Быстрая санитарная проверка на примерах ─────────────────────────────
from wav2vec2decoder import Wav2Vec2Decoder, test

dec = Wav2Vec2Decoder(lm_model_path=LM_PATH, beam_width=BEAM_WIDTH, alpha=0.1, beta=0.5)

test(dec, 'examples/sample1.wav',
     'if you are generous here is a fitting opportunity for the exercise of your magnanimity '
     'if you are proud here am i your rival ready to acknowledge myself your debtor for an act '
     'of the most noble forbearance')
test(dec, 'examples/sample6.wav', 'at this time all participants are in a listen only mode')

In [ ]:
# ── 6. Полная оценка (Tasks 1–7b) ──────────────────────────────────────────
!python evaluate.py \
    --data-root . \
    --lm {LM_PATH} \
    --beam-width {BEAM_WIDTH} \
    --max-samples {MAX_SAMPLES}

In [ ]:
# ── 7. Показать сохранённые графики ───────────────────────────────────────
from IPython.display import Image, display
import glob

for path in sorted(glob.glob('results/*.png')):
    print(f'\n--- {os.path.basename(path)} ---')
    display(Image(path))

In [ ]:
# ── 8. (Task 8) Обучение финансового KenLM ─────────────────────────────────
# Только в Colab (Linux): строим KenLM CLI и обучаем 3-gram на earnings22_train/corpus.txt

!apt-get install -q cmake libboost-all-dev
!git clone --depth=1 https://github.com/kpu/kenlm /tmp/kenlm_build
!mkdir -p /tmp/kenlm_build/build
!cmake -S /tmp/kenlm_build -B /tmp/kenlm_build/build -DCMAKE_BUILD_TYPE=Release
!make -C /tmp/kenlm_build/build -j2 lmplz build_binary

!mkdir -p lm
!/tmp/kenlm_build/build/bin/lmplz -o 3 --discount_fallback \
    < data/earnings22_train/corpus.txt \
    > /tmp/financial-3gram.arpa

# Используем как plain .arpa (kenlm Python не требует gzip в Colab)
!cp /tmp/financial-3gram.arpa lm/financial-3gram.arpa
print('Financial LM ready: lm/financial-3gram.arpa')

In [ ]:
# ── 9. (Task 9) Сравнение LibriSpeech LM vs Financial LM ──────────────────
import csv, jiwer
from wav2vec2decoder import Wav2Vec2Decoder
import torchaudio

def load_manifest(path, max_n=MAX_SAMPLES):
    rows = []
    with open(path) as f:
        for r in csv.DictReader(f):
            rows.append((r['path'], r['text']))
            if len(rows) >= max_n:
                break
    return rows

def eval_set(decoder, samples, method):
    hyps, refs = [], []
    for path, ref in samples:
        audio, _ = torchaudio.load(path)
        hyps.append(decoder.decode(audio, method))
        refs.append(ref)
    return jiwer.wer(refs, hyps), jiwer.cer(refs, hyps)

ls  = load_manifest('data/librispeech_test_other/manifest.csv')
e22 = load_manifest('data/earnings22_test/manifest.csv')

results = []
for lm_name, lm_path in [('LibriSpeech 3-gram', 'lm/3-gram.pruned.1e-7.arpa'),
                           ('Financial 3-gram',  'lm/financial-3gram.arpa')]:
    for method in ['beam_lm', 'beam_lm_rescore']:
        dec = Wav2Vec2Decoder(lm_model_path=lm_path, beam_width=BEAM_WIDTH, alpha=0.1, beta=0.5)
        wer_ls, cer_ls   = eval_set(dec, ls,  method)
        wer_e22, cer_e22 = eval_set(dec, e22, method)
        row = {'LM': lm_name, 'Method': method,
               'LS WER': f'{wer_ls:.4f}', 'LS CER': f'{cer_ls:.4f}',
               'E22 WER': f'{wer_e22:.4f}', 'E22 CER': f'{cer_e22:.4f}'}
        results.append(row)
        print(row)

import pandas as pd
pd.DataFrame(results).to_csv('results/task9_lm_comparison.csv', index=False)
print('Saved: results/task9_lm_comparison.csv')